In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__notebook__.ipynb
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__output__.json
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__results__.html
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/custom.css
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_test.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_train.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_val.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/y_test.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/y_train.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/y_val.csv


In [7]:
from pathlib import Path

DATA_DIR = Path("/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")

y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()
y_val = pd.read_csv(DATA_DIR / "y_val.csv").squeeze()
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze()

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(28831, 19)
(6178, 19)
(6179, 19)
(28831,)
(6178,)
(6179,)


In [8]:
v2_drop=["emp.var.rate", "euribor3m"]
X_train.drop(columns=v2_drop, inplace=True)
X_val.drop(columns=v2_drop, inplace=True)
X_test.drop(columns=v2_drop, inplace=True)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(28831, 17)
(6178, 17)
(6179, 17)


In [9]:
y_train.value_counts()

y
0    25583
1     3248
Name: count, dtype: int64

In [10]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Identify numerical and categorical columns
cat_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

num_cols = X_train.select_dtypes(
    include=np.number
).columns.tolist()

# Numerical preprocessing
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical preprocessing
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        drop='first'
    ))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Fit on train; transform validation and test
X_train_encoded = preprocessor.fit_transform(X_train)
X_val_encoded = preprocessor.transform(X_val)
X_test_encoded = preprocessor.transform(X_test)

print("Train:", X_train_encoded.shape)
print("Validation:", X_val_encoded.shape)
print("Test:", X_test_encoded.shape)

Train: (28831, 50)
Validation: (6178, 50)
Test: (6179, 50)


In [11]:
from xgboost import XGBClassifier

In [12]:
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [13]:
def ks_scorer(estimator, X, y):
    
    y_prob = estimator.predict_proba(X)[:, 1]

    df = pd.DataFrame({
        "actual": np.array(y),
        "probability": y_prob
    })

    df = df.sort_values(
        "probability",
        ascending=False
    ).reset_index(drop=True)

    total_positives = df["actual"].sum()
    total_negatives = len(df) - total_positives

    df["cum_positives"] = df["actual"].cumsum()
    df["cum_negatives"] = (1 - df["actual"]).cumsum()

    df["cum_positive_rate"] = (
        df["cum_positives"] / total_positives
    )

    df["cum_negative_rate"] = (
        df["cum_negatives"] / total_negatives
    )

    df["KS"] = abs(
        df["cum_positive_rate"]
        - df["cum_negative_rate"]
    )

    return df["KS"].max()

In [14]:
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 50, 200
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 3, 7
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 10
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),

        "gamma": trial.suggest_float(
            "gamma", 0, 5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 0.1, 10,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 10,
            log=True
        )
        
        # "scale_pos_weight": trial.suggest_float(
        #     "scale_pos_weight", 5.0, 9.0
        # )
    }

    model = XGBClassifier(
        **params,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
        tree_method="hist"
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "roc_auc": "roc_auc",
            "ks": ks_scorer
        },
        return_train_score=True,
        n_jobs=-1
    )

    train_auc = scores["train_roc_auc"].mean()
    cv_auc = scores["test_roc_auc"].mean()
    train_ks = scores["train_ks"].mean()
    cv_ks = scores["test_ks"].mean()

    trial.set_user_attr("Train ROC-AUC", train_auc)
    trial.set_user_attr("CV ROC-AUC", cv_auc)
    trial.set_user_attr("Train KS", train_ks)
    trial.set_user_attr("CV KS", cv_ks)

    return cv_auc

In [15]:
os.makedirs("/kaggle/working/outputs", exist_ok=True)

log_path = "/kaggle/working/outputs/optuna_xgb_trials_06.csv"

def log_optuna_trial(study, trial):

    rows = []

    for t in study.trials:

        row = {
            "Trial": t.number,
            "Train ROC-AUC": t.user_attrs.get("Train ROC-AUC", np.nan),
            "CV ROC-AUC": t.user_attrs.get("CV ROC-AUC", np.nan),
            "Train KS": t.user_attrs.get("Train KS", np.nan),
            "CV KS": t.user_attrs.get("CV KS", np.nan),
            "AUC Gap": t.user_attrs.get("AUC Gap",np.nan),
            "KS Gap": t.user_attrs.get("KS Gap",np.nan),
            "Best CV ROC-AUC": study.best_value,
            "Best Trial": study.best_trial.number,
            "Is Best": (t.number == study.best_trial.number)
        }

        row.update(t.params)
        rows.append(row)
    pd.DataFrame(rows).sort_values(
        "Trial"
    ).to_csv(
        log_path,
        index=False
    )

In [16]:
!pip install -q optuna
import optuna

In [17]:
study = optuna.create_study(
    direction="maximize",
    study_name="xgboost_roc_auc"
)

study.optimize(
    objective,
    n_trials=150,
    callbacks=[log_optuna_trial]
)

[I 2026-11-27 06:59:25,461] A new study created in memory with name: xgboost_roc_auc
[I 2026-11-27 06:59:30,834] Trial 0 finished with value: 0.7911930353818086 and parameters: {'n_estimators': 123, 'learning_rate': 0.038284303464931016, 'max_depth': 4, 'min_child_weight': 10, 'subsample': 0.900378069636151, 'colsample_bytree': 0.8626500861955995, 'gamma': 2.7180647171518215, 'reg_alpha': 0.9536264518298658, 'reg_lambda': 0.009632950346544258}. Best is trial 0 with value: 0.7911930353818086.
[I 2026-11-27 06:59:32,852] Trial 1 finished with value: 0.7901830695105134 and parameters: {'n_estimators': 99, 'learning_rate': 0.09946500420005473, 'max_depth': 4, 'min_child_weight': 7, 'subsample': 0.6141324651628308, 'colsample_bytree': 0.6570863738882945, 'gamma': 4.8736001603295, 'reg_alpha': 1.2186189768306017, 'reg_lambda': 0.16474268502141304}. Best is trial 0 with value: 0.7911930353818086.
[I 2026-11-27 06:59:34,801] Trial 2 finished with value: 0.7914928647524789 and parameters: {'n_e

In [18]:
optuna_results = pd.read_csv(
    "/kaggle/working/outputs/optuna_xgb_trials_06.csv"
)

optuna_results.round(4)

,Trial,Train ROC-AUC,CV ROC-AUC,Train KS,CV KS,AUC Gap,KS Gap,Best CV ROC-AUC,Best Trial,Is Best,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda
0,0,0.8041,0.7912,0.4889,0.4854,NaN,NaN,0.7943,127,False,123,0.0383,4,10,0.9004,0.8627,2.7181,0.9536,0.0096
1,1,0.7990,0.7902,0.4873,0.4855,NaN,NaN,0.7943,127,False,99,0.0995,4,7,0.6141,0.6571,4.8736,1.2186,0.1647
2,2,0.8005,0.7915,0.4912,0.4872,NaN,NaN,0.7943,127,False,77,0.0607,5,9,0.8977,0.6610,4.2045,1.2555,0.0011
3,3,0.7939,0.7890,0.4822,0.4814,NaN,NaN,0.7943,127,False,83,0.0327,4,5,0.9094,0.6138,3.3451,5.5179,0.2654
4,4,0.8479,0.7911,0.5339,0.4847,NaN,NaN,0.7943,127,False,181,0.0573,6,8,0.6622,0.8242,0.9225,0.5751,0.0091
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,145,0.8162,0.7939,0.4978,0.4882,NaN,NaN,0.7943,127,False,132,0.0275,6,4,0.7372,0.6240,2.3572,0.2277,0.0278
146,146,0.8150,0.7933,0.4976,0.4899,NaN,NaN,0.7943,127,False,128,0.0243,6,3,0.7507,0.6319,2.3888,0.1440,0.0208
147,147,0.8188,0.7940,0.4992,0.4888,NaN,NaN,0.7943,127,False,123,0.0266,6,3,0.7221,0.6443,2.0695,0.2190,0.0100
148,148,0.8174,0.7941,0.4998,0.4892,NaN,NaN,0.7943,127,False,127,0.0285,6,3,0.7651,0.6101,2.4109,0.1764,0.0181


In [19]:
print("Best Trial:", study.best_trial.number)
print("Best CV ROC-AUC:", study.best_value)
print("Best Parameters:")

for param, value in study.best_params.items():
    print(f"{param}: {value}")

Best Trial: 127
Best CV ROC-AUC: 0.7943365646277802
Best Parameters:
n_estimators: 136
learning_rate: 0.02866598475968572
max_depth: 6
min_child_weight: 3
subsample: 0.7405242794892563
colsample_bytree: 0.6068049158355833
gamma: 2.2636758092874487
reg_alpha: 0.3003442024029645
reg_lambda: 0.04151517100622987


In [28]:
optuna_results[optuna_results["Trial"] == 127]

,Trial,Train ROC-AUC,CV ROC-AUC,Train KS,CV KS,AUC Gap,KS Gap,Best CV ROC-AUC,Best Trial,Is Best,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda
127,127,0.818409,0.794337,0.500208,0.488477,NaN,NaN,0.794337,127,True,136,0.028666,6,3,0.740524,0.606805,2.263676,0.300344,0.041515


In [20]:
best_xgb = XGBClassifier(
    **study.best_params,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
    tree_method="hist"
)

best_xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_xgb)
])

best_xgb_pipeline.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['age', 'campaign', 'pdays',
                                                   'previous', 'cons.price.idx',
                                                   'cons.conf.idx',
                                                   'nr.employed']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ig...
                               gamma=2.2636758092874487, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.02866598475968572, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=6,
                               max_leaves=None, min_child_weight=3, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=136, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [21]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

y_val_pred = best_xgb_pipeline.predict(X_val)
y_val_prob = best_xgb_pipeline.predict_proba(X_val)[:, 1]

tuned_xgb_results = {
    "Model": "XGBoost Tuned",
    "Accuracy": accuracy_score(y_val, y_val_pred),
    "Precision": precision_score(y_val, y_val_pred),
    "Recall": recall_score(y_val, y_val_pred),
    "F1": f1_score(y_val, y_val_pred),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob)
}

pd.DataFrame([tuned_xgb_results]).round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,XGBoost Tuned,0.9011,0.6721,0.2385,0.3521,0.8083


In [27]:
from optuna.visualization import (
    plot_optimization_history,
    plot_parallel_coordinate
)

# Optimization history
fig1 = plot_optimization_history(study)
fig1.show()

# Parallel coordinate plot
fig2 = plot_parallel_coordinate(study)
fig2.show()

In [22]:
# models_cv = {
#     "XGBoost": (xgb, X_train_encoded)
# }

# cv_results = []

# for name, (model, X_train_model) in models_cv.items():

#     scores = cross_validate(
#         model,
#         X_train_model,
#         y_train,
#         cv=cv,
#         scoring=[
#             "roc_auc",
#             "precision",
#             "recall",
#             "f1"
#         ],
#         n_jobs=-1
#     )

#     cv_results.append({
#         "Model": name,
#         "CV ROC-AUC": scores["test_roc_auc"].mean(),
#         "CV Precision": scores["test_precision"].mean(),
#         "CV Recall": scores["test_recall"].mean(),
#         "CV F1": scores["test_f1"].mean()
#     })

# cv_results_df = pd.DataFrame(cv_results)

# cv_results_df.round(4)

In [23]:
# xgb.fit(X_train_encoded, y_train)

In [24]:
# xgb_prob = xgb.predict_proba(X_val_encoded)[:, 1]

In [25]:
# from sklearn.metrics import (
#     accuracy_score,
#     precision_score,
#     recall_score,
#     f1_score,
#     roc_auc_score
# )
# models = {
#     "XGBoost": (xgb, X_val_encoded)
# }

# results = []

# for name, (model, X_val_model) in models.items():

#     y_pred = model.predict(X_val_model)
#     y_prob = model.predict_proba(X_val_model)[:, 1]

#     results.append({
#         "Model": name,
#         "Accuracy": accuracy_score(y_val, y_pred),
#         "Precision": precision_score(y_val, y_pred),
#         "Recall": recall_score(y_val, y_pred),
#         "F1": f1_score(y_val, y_pred),
#         "ROC-AUC": roc_auc_score(y_val, y_prob)
#     })